# Baseline Model Evaluation for PsyQA Dataset
## Fine-tuning Multilingual LLMs using LoRA for Mental Health Context

This notebook evaluates baseline models on the PsyQA dataset:
- **General LLMs**: LLaMA-3, Gemma, Mistral
- **Multilingual Models**: mT5, BLOOMZ, XGLM
- **Domain-Tuned**: Mental-Mistral

**Evaluation Metrics**: ROUGE-L, BLEU-4, BERTScore, BERT F1

## 1. Installation and Setup

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch datasets evaluate rouge-score nltk bert-score sacrebleu sentencepiece protobuf langdetect bitsandbytes

In [ ]:
# Import libraries
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Transformers and model libraries
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
    logging
)
logging.set_verbosity_error()

# Evaluation metrics
from evaluate import load
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
from langdetect import detect, LangDetectException

# ====================================================
# METRIC COMPUTATION CONTROL
# Set RECOMPUTE_METRICS = True to re-enable ROUGE/BLEU/BERTScore.
# When False: legacy metrics are SKIPPED; existing CSV/JSON
# files remain the source of truth for those columns.
# Only the new empathy metric is computed on every run.
# ====================================================
RECOMPUTE_METRICS = False

print(f"RECOMPUTE_METRICS = {RECOMPUTE_METRICS}")
if not RECOMPUTE_METRICS:
    print("Legacy metrics (ROUGE, BLEU, BERTScore) will NOT be recomputed.")
    print("Existing CSV/JSON outputs preserve those values.")


In [ ]:
# HuggingFace Token Input (Runtime)
from getpass import getpass

HF_TOKEN = getpass("Enter your HuggingFace token: ")

# Login to HuggingFace
from huggingface_hub import login
login(token=HF_TOKEN)
print("✓ Successfully logged in to HuggingFace")

## 2. Load and Preprocess Dataset

In [ ]:
# Load PsyQA dataset
def load_psyqa_data(file_path: str, max_samples: int = 50):
    """
    Load PsyQA dataset from JSON file
    Args:
        file_path: Path to PsyQA_example.json
        max_samples: Maximum number of samples to load (for testing)
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Limit samples for faster evaluation
    data = data[:max_samples]
    
    # Extract relevant fields
    processed_data = []
    for item in data:
        processed_item = {
            'question': item['question'],
            'description': item.get('description', ''),
            'keywords': item.get('keywords', ''),
            'reference_answer': item['answers'][0]['answer_text'] if item['answers'] else '',
            'questionID': item['questionID']
        }
        processed_data.append(processed_item)
    
    return processed_data

# Load data
data_path = 'PsyQA_example.json'
psyqa_data = load_psyqa_data(data_path, max_samples=50)

print(f"Loaded {len(psyqa_data)} samples from PsyQA dataset")
print(f"\nExample:")
print(f"Question: {psyqa_data[0]['question']}")
print(f"Description: {psyqa_data[0]['description'][:100]}...")
print(f"Reference Answer: {psyqa_data[0]['reference_answer'][:100]}...")

In [ ]:
# Create input prompts
def create_prompt(question: str, description: str) -> str:
    """
    Create a prompt for mental health Q&A
    """
    if description:
        return f"问题：{question}\n描述：{description}\n回答："
    else:
        return f"问题：{question}\n回答："

# Add prompts to data
for item in psyqa_data:
    item['prompt'] = create_prompt(item['question'], item['description'])

print("Sample prompt:")
print(psyqa_data[0]['prompt'])

## 3. Model Configuration

In [ ]:
# Model configurations
MODEL_CONFIGS = {
    # General LLMs
    "LLaMA-3-8B": {
        "model_name": "meta-llama/Meta-Llama-3-8B-Instruct",
        "type": "causal",
        "category": "General LLM"
    },
    "Gemma-7B": {
        "model_name": "google/gemma-7b-it",
        "type": "causal",
        "category": "General LLM"
    },
    "Mistral-7B": {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
        "type": "causal",
        "category": "General LLM"
    },
    
    # Multilingual Models
    "mT5-Base": {
        "model_name": "google/mt5-base",
        "type": "seq2seq",
        "category": "Multilingual"
    },
    "BLOOMZ-7B": {
        "model_name": "bigscience/bloomz-7b1",
        "type": "causal",
        "category": "Multilingual"
    },
    "XGLM-7.5B": {
        "model_name": "facebook/xglm-7.5B",
        "type": "causal",
        "category": "Multilingual"
    },
    
    # Domain-Tuned
    "Mental-Mistral": {
        "model_name": "Isaachhe/Mental-Mistral-7b",
        "type": "causal",
        "category": "Domain-Tuned"
    }
}

print("Models to evaluate:")
for name, config in MODEL_CONFIGS.items():
    print(f"  • {name} ({config['category']})")

## 4. Model Loading and Inference Functions

In [ ]:
def load_model_and_tokenizer(model_name: str, model_type: str):
    """
    Load model and tokenizer based on type.
    4-bit quantization is used for causal models to reduce VRAM from ~16 GB to ~5 GB,
    preventing CPU/disk offloading that makes inference extremely slow.
    """
    print(f"Loading {model_name}...")
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HF_TOKEN,
        trust_remote_code=True
    )
    
    # Set pad token if not exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 4-bit quantization config — keeps large causal models (7B/8B) fully on GPU
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    # Load model based on type
    if model_type == "seq2seq":
        # seq2seq models (e.g. mT5) are small enough — no quantization needed
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:  # causal
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=HF_TOKEN,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    
    print(f"✓ Model loaded successfully")
    return model, tokenizer


def generate_response(model, tokenizer, prompt: str, model_type: str, max_length: int = 256) -> str:
    """
    Generate response from model
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    if model_type == "seq2seq":
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:  # causal - remove the prompt
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = full_output[len(prompt):].strip()
    
    return response


def evaluate_model(model_name: str, model_type: str, data: List[Dict]) -> Dict[str, List[str]]:
    """
    Run inference on all samples
    """
    model, tokenizer = load_model_and_tokenizer(model_name, model_type)
    
    predictions = []
    references = []
    
    print(f"\nGenerating responses...")
    for item in tqdm(data):
        prompt = item['prompt']
        reference = item['reference_answer']
        
        prediction = generate_response(model, tokenizer, prompt, model_type)
        
        predictions.append(prediction)
        references.append(reference)
    
    # Clear GPU memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    
    return {
        'predictions': predictions,
        'references': references
    }

print("Model loading and inference functions defined ✓")

## 5. Evaluation Metrics

In [ ]:
def calculate_rouge_l(predictions: List[str], references: List[str]) -> float:
    """
    Calculate ROUGE-L score
    """
    rouge = load('rouge')
    results = rouge.compute(
        predictions=predictions,
        references=references,
        rouge_types=['rougeL']
    )
    return results['rougeL'] * 100  # Convert to percentage


def calculate_bleu_4(predictions: List[str], references: List[str]) -> float:
    """
    Calculate BLEU-4 score
    """
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    
    for pred, ref in zip(predictions, references):
        # Tokenize (character-level for Chinese)
        pred_tokens = list(pred)
        ref_tokens = [list(ref)]
        
        score = sentence_bleu(
            ref_tokens,
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
        bleu_scores.append(score)
    
    return np.mean(bleu_scores) * 100  # Convert to percentage


def calculate_bert_score(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """
    Calculate BERTScore (Precision, Recall, F1)
    """
    P, R, F1 = bert_score(
        predictions,
        references,
        lang='zh',  # Chinese language
        verbose=False,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    
    return {
        'precision': P.mean().item() * 100,
        'recall': R.mean().item() * 100,
        'f1': F1.mean().item() * 100
    }


def compute_all_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """
    Compute all evaluation metrics
    """
    print("  Computing ROUGE-L...")
    rouge_l = calculate_rouge_l(predictions, references)
    
    print("  Computing BLEU-4...")
    bleu_4 = calculate_bleu_4(predictions, references)
    
    print("  Computing BERTScore...")
    bert_scores = calculate_bert_score(predictions, references)
    
    return {
        'ROUGE-L': rouge_l,
        'BLEU-4': bleu_4,
        'BERTScore-P': bert_scores['precision'],
        'BERTScore-R': bert_scores['recall'],
        'BERTScore-F1': bert_scores['f1']
    }

print("Evaluation metrics functions defined ✓")

## 6. Run Baseline Evaluation

In [ ]:
# Main evaluation loop
results = {}

for model_name, config in MODEL_CONFIGS.items():
    print(f"\n{'='*80}")
    print(f"Evaluating: {model_name} ({config['category']})")
    print(f"{'='*80}")
    
    try:
        # Generate predictions
        outputs = evaluate_model(
            config['model_name'],
            config['type'],
            psyqa_data
        )
        
        # NOTE: Legacy metrics are skipped when RECOMPUTE_METRICS=False.
        # The function definitions above remain available if needed.
        if RECOMPUTE_METRICS:
            print("\nComputing metrics...")
            metrics = compute_all_metrics(
                outputs['predictions'],
                outputs['references']
            )
        else:
            metrics = {
                'ROUGE-L': float('nan'), 'BLEU-4': float('nan'),
                'BERTScore-P': float('nan'), 'BERTScore-R': float('nan'),
                'BERTScore-F1': float('nan')
            }
            print("\nSkipping legacy metric recomputation (RECOMPUTE_METRICS=False).")
        
        # Store results
        results[model_name] = {
            'category': config['category'],
            'metrics': metrics,
            # All predictions stored for empathy scoring
            'predictions': outputs['predictions'],
            'references':  outputs['references'],
            'sample_outputs': [
                {
                    'question': psyqa_data[i]['question'],
                    'prediction': outputs['predictions'][i],
                    'reference': outputs['references'][i]
                }
                for i in range(min(3, len(psyqa_data)))
            ]
        }
        
        print(f"\n✓ {model_name} evaluation complete")
        if RECOMPUTE_METRICS:
            print(f"  ROUGE-L: {metrics['ROUGE-L']:.2f}")
            print(f"  BLEU-4: {metrics['BLEU-4']:.2f}")
            print(f"  BERTScore F1: {metrics['BERTScore-F1']:.2f}")
        
    except Exception as e:
        print(f"\n✗ Error evaluating {model_name}: {str(e)}")
        results[model_name] = {
            'category': config['category'],
            'error': str(e)
        }

print(f"\n{'='*80}")
print("All evaluations complete!")
print(f"{'='*80}")

## 7. Results Comparison and Analysis

In [ ]:
# Create results DataFrame
results_data = []

for model_name, result in results.items():
    if 'metrics' in result:
        row = {
            'Model': model_name,
            'Category': result['category'],
            'ROUGE-L': result['metrics']['ROUGE-L'],
            'BLEU-4': result['metrics']['BLEU-4'],
            'BERTScore-P': result['metrics']['BERTScore-P'],
            'BERTScore-R': result['metrics']['BERTScore-R'],
            'BERTScore-F1': result['metrics']['BERTScore-F1']
        }
        results_data.append(row)

results_df = pd.DataFrame(results_data)

# Sort by BERTScore F1 (primary metric)
results_df = results_df.sort_values('BERTScore-F1', ascending=False)

print("\n" + "="*100)
print("BASELINE MODEL COMPARISON")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

In [ ]:
# Styled DataFrame for better visualization
styled_df = results_df.style.background_gradient(
    subset=['ROUGE-L', 'BLEU-4', 'BERTScore-F1'],
    cmap='YlGn'
).format({
    'ROUGE-L': '{:.2f}',
    'BLEU-4': '{:.2f}',
    'BERTScore-P': '{:.2f}',
    'BERTScore-R': '{:.2f}',
    'BERTScore-F1': '{:.2f}'
})

display(styled_df)

In [ ]:
# Best model by category
print("\nBEST MODEL BY CATEGORY:")
print("="*100)

for category in results_df['Category'].unique():
    category_df = results_df[results_df['Category'] == category]
    best_model = category_df.iloc[0]
    
    print(f"\n{category}:")
    print(f"  🏆 {best_model['Model']}")
    print(f"     ROUGE-L: {best_model['ROUGE-L']:.2f}")
    print(f"     BLEU-4: {best_model['BLEU-4']:.2f}")
    print(f"     BERTScore F1: {best_model['BERTScore-F1']:.2f}")

print("\n" + "="*100)
print("\nOVERALL BEST MODEL:")
best_overall = results_df.iloc[0]
print(f"  🥇 {best_overall['Model']} ({best_overall['Category']})")
print(f"     ROUGE-L: {best_overall['ROUGE-L']:.2f}")
print(f"     BLEU-4: {best_overall['BLEU-4']:.2f}")
print(f"     BERTScore F1: {best_overall['BERTScore-F1']:.2f}")
print("="*100)

## 8. Sample Outputs Inspection

In [ ]:
# Display sample outputs from best model
best_model_name = results_df.iloc[0]['Model']
sample_outputs = results[best_model_name]['sample_outputs']

print(f"\nSAMPLE OUTPUTS FROM BEST MODEL: {best_model_name}")
print("="*100)

for i, sample in enumerate(sample_outputs, 1):
    print(f"\nExample {i}:")
    print(f"Question: {sample['question']}")
    print(f"\nModel Prediction:\n{sample['prediction']}")
    print(f"\nReference Answer:\n{sample['reference']}")
    print("-" * 100)

In [ ]:
# Compare outputs across all models for one question
question_idx = 0
question = psyqa_data[question_idx]['question']

print(f"\nCOMPARISON OF ALL MODELS FOR QUESTION:")
print(f"'{question}'")
print("="*100)

for model_name, result in results.items():
    if 'sample_outputs' in result:
        print(f"\n{model_name}:")
        print(f"{result['sample_outputs'][question_idx]['prediction']}")
        print("-" * 100)

print(f"\nReference Answer:")
print(f"{psyqa_data[question_idx]['reference_answer']}")
print("="*100)

## 9. Save Results

In [ ]:
# Save results to CSV
results_df.to_csv('baseline_evaluation_results.csv', index=False)
print("✓ Results saved to baseline_evaluation_results.csv")

# Save detailed results to JSON
with open('baseline_evaluation_detailed.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("✓ Detailed results saved to baseline_evaluation_detailed.json")

## Empathy Scoring (New Metric)

Computes `empathy_score` using `j-hartmann/emotion-english-distilroberta-base`.
- Scores each response across 7 emotions: anger, disgust, fear, joy, neutral, sadness, surprise.
- `empathy_score` = probability of **joy** (warmth/positive engagement — standard proxy for empathic text).
- **Only English responses** are scored (detected via `langdetect`).
- Non-English responses receive `NaN`.
- ROUGE / BLEU / BERTScore are **not** recomputed here.

In [ ]:
!pip install -q langdetect

In [ ]:
# ====================================================
# EMPATHY SCORING FUNCTION
# Model: j-hartmann/emotion-english-distilroberta-base
# empathy_score = joy probability (warmth/positive engagement)
# ====================================================

def compute_empathy_scores(texts, batch_size=16):
    """
    Score a list of texts for empathy (0.0 – 1.0).

    Uses j-hartmann/emotion-english-distilroberta-base to classify 7 emotions.
    empathy_score is the 'joy' probability — the standard NLP proxy for
    warm, empathic responses in mental health / counseling contexts.

    Only English-detected texts are scored.
    Non-English and empty strings receive float('nan').

    Args:
        texts (list[str]): Generated responses to score.
        batch_size (int): Number of texts per inference batch.

    Returns:
        list[float]: Per-text empathy score (joy probability, or NaN).
    """
    from transformers import pipeline as hf_pipeline

    device = 0 if torch.cuda.is_available() else -1
    scores = [float('nan')] * len(texts)

    # --- language detection: keep only English ---
    english_indices, english_texts = [], []
    for i, text in enumerate(texts):
        if not text or not str(text).strip():
            continue
        try:
            lang = detect(str(text))
        except LangDetectException:
            lang = 'unknown'
        if lang == 'en':
            english_indices.append(i)
            english_texts.append(str(text)[:512])

    if not english_texts:
        print("No English responses detected — all empathy_score values will be NaN.")
        return scores

    print(f"Loading empathy classifier on device={device} ...")
    empathy_clf = hf_pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=None,          # return scores for all 7 emotion labels
        device=device,
        batch_size=batch_size,
    )

    print(f"Scoring {len(english_texts)} / {len(texts)} English responses "
          f"(batch_size={batch_size}) ...")
    raw = empathy_clf(english_texts, truncation=True, max_length=512)

    for idx, result in zip(english_indices, raw):
        # result is a list of {'label': str, 'score': float} for all 7 emotions
        label_scores = {item['label']: item['score'] for item in result}
        # joy = warmth / positive engagement — best single-label proxy for empathy
        scores[idx] = label_scores.get('joy', float('nan'))

    del empathy_clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return scores

print("✓ compute_empathy_scores() defined")

In [ ]:
# ====================================================
# COMPUTE EMPATHY SCORES AND SAVE UPDATED RESULTS
# ====================================================

PREDICTIONS_SAVE_PATH = 'baseline_evaluation_predictions.csv'
SUMMARY_CSV           = 'baseline_evaluation_results.csv'

# --- Collect predictions from the current run ---
pred_rows = []
for _model_name, _result in results.items():
    if 'error' in _result:
        continue
    _preds = _result.get('predictions', [])
    _refs  = _result.get('references',  [])
    for _pred, _ref in zip(_preds, _refs):
        pred_rows.append({'model': _model_name, 'prediction': _pred, 'reference': _ref})

predictions_df = pd.DataFrame(pred_rows) if pred_rows else pd.DataFrame(
    columns=['model', 'prediction', 'reference'])

# --- Merge any previously saved empathy scores ---
if os.path.exists(PREDICTIONS_SAVE_PATH):
    _prior = pd.read_csv(PREDICTIONS_SAVE_PATH)
    if 'empathy_score' in _prior.columns and not predictions_df.empty:
        print(f"Merging prior empathy scores from {PREDICTIONS_SAVE_PATH} ...")
        predictions_df = predictions_df.merge(
            _prior[['model', 'prediction', 'empathy_score']],
            on=['model', 'prediction'], how='left')
    elif predictions_df.empty and 'empathy_score' in _prior.columns:
        print(f"No new predictions generated. Loading all from {PREDICTIONS_SAVE_PATH} ...")
        predictions_df = _prior
    else:
        predictions_df['empathy_score'] = float('nan')
else:
    predictions_df['empathy_score'] = float('nan')

# --- Score any rows that still lack an empathy score ---
_mask = predictions_df['empathy_score'].isna()
if _mask.any():
    _new_scores = compute_empathy_scores(
        predictions_df.loc[_mask, 'prediction'].tolist(), batch_size=16)
    predictions_df.loc[_mask, 'empathy_score'] = _new_scores
    print(f"Scored {_mask.sum()} new predictions.")
else:
    print("All predictions already have empathy scores — nothing new to compute.")

# --- Save per-response file ---
predictions_df.to_csv(PREDICTIONS_SAVE_PATH, index=False)
print(f"Saved {len(predictions_df)} rows → {PREDICTIONS_SAVE_PATH}")

# --- Update summary CSV (preserve old metric columns, add empathy_score) ---
_mean_empathy = (
    predictions_df.groupby('model')['empathy_score']
    .mean().rename('empathy_score'))

if os.path.exists(SUMMARY_CSV):
    _summary = pd.read_csv(SUMMARY_CSV)
else:
    _rows = []
    for _mn, _r in results.items():
        if 'metrics' in _r:
            _row = {'Model': _mn, 'Category': _r['category']}
            _row.update(_r['metrics'])
            _rows.append(_row)
    _summary = pd.DataFrame(_rows) if _rows else pd.DataFrame()

if not _summary.empty:
    _summary['empathy_score'] = _summary['Model'].map(_mean_empathy)
    _summary.to_csv(SUMMARY_CSV, index=False)
    print(f"Updated {SUMMARY_CSV} with empathy_score column.")
    _cols = ['Model'] + [c for c in _summary.columns if c != 'Model']
    print(_summary[[c for c in _cols if c in _summary.columns]].to_string(index=False))


## 10. Visualization

In [ ]:
# Install matplotlib if needed
!pip install -q matplotlib seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Baseline Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['ROUGE-L', 'BLEU-4', 'BERTScore-F1', 'BERTScore-P']
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax = axes[idx // 2, idx % 2]
    
    # Sort by metric
    plot_df = results_df.sort_values(metric, ascending=True)
    
    # Create bar plot
    bars = ax.barh(plot_df['Model'], plot_df[metric], color=color, alpha=0.7)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.2f}', ha='left', va='center', fontweight='bold')
    
    ax.set_xlabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Scores', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to baseline_comparison.png")

In [ ]:
# Grouped by category
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data
categories = results_df['Category'].unique()
x = np.arange(len(results_df))
width = 0.2

# Plot metrics
metrics = ['ROUGE-L', 'BLEU-4', 'BERTScore-F1']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    offset = (i - 1) * width
    ax.bar(x + offset, results_df[metric], width, label=metric, color=color, alpha=0.8)

ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Multi-Metric Comparison Across Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('multi_metric_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to multi_metric_comparison.png")

## Summary

This notebook evaluated multiple baseline models on the PsyQA mental health Q&A dataset:

**Models Tested:**
- General LLMs: LLaMA-3, Gemma, Mistral
- Multilingual: mT5, BLOOMZ, XGLM
- Domain-Tuned: Mental-Mistral

**Evaluation Metrics:**
- ROUGE-L: Measures overlap of longest common subsequences
- BLEU-4: Measures n-gram precision
- BERTScore: Semantic similarity using contextual embeddings
- BERT F1: Harmonic mean of precision and recall

The results provide a comprehensive baseline for comparison before fine-tuning with LoRA.